In [ ]:
from libraries import *
from parameters import *
from Utilities import *

In [ ]:
os.getcwd()
os.chdir(projectDir)

In [ ]:
%load_ext rpy2.ipython

In [ ]:
adata = sc.read("./Data/adataALL.h5ad")

In [ ]:
nepcSignatures = pd.DataFrame(pd.read_csv("./TextFiles/NEPC_adenocarcinoma_markerGenes.csv", index_col=0))
geneSignatures = pd.DataFrame(pd.read_csv("./TextFiles/Human_NE_signatures.csv"))

selectedGenes = list(set(list(geneSignatures["High in NE"]) + list(geneSignatures["Low in NE"]) + 
         list(nepcSignatures["adenocarcinoma"]) + list(nepcSignatures["NEPC"])))

myGeneList = [x for x in selectedGenes if x != ' ']
myGeneList = [x for x in myGeneList if x !="nan"]
myGeneList = [x for x in myGeneList if not isinstance(x, float)]

myGeneList = [x.replace(".","-") for x in myGeneList]

myGeneList = [x for x in myGeneList if x in adata.var_names]
    

In [ ]:
adata = adata[:,myGeneList]

In [ ]:
adata.obs["guide_num"].value_counts()

In [ ]:
adatasingles = adata[(adata.obs["guide_num"] == 1) | (adata.obs["guide_num"] ==100),]

In [ ]:
adatasingles.obs["guide_num"].value_counts()

In [ ]:
targetCells = adatasingles[adatasingles.obs['leiden'] == '15',:]

In [ ]:
adataKOScreen = adatasingles[adatasingles.obs["guide_id"] != "None",:]

In [ ]:
KOIds = adataKOScreen.obs["guide_id"].unique()

In [ ]:
myDF = pd.DataFrame({"KO":KOIds, "OTDist":0})
myDF.index=myDF.KO
myDF

In [ ]:
import ot
def ot_distance(exp1, exp2, N=None):
    if N is not None:
        s1 = np.random.choice(exp1.shape[0], N, replace=True)  
        exp1 = exp1[s1,:]
        s2 = np.random.choice(exp2.shape[0], N, replace=True)
        exp2 = exp2[s2,:]
    M = ot.dist(exp1, exp2)
    #M /= M.max()
    
    a = np.ones(len(exp1)) / len(exp1)
    b = np.ones(len(exp2)) / len(exp2)
    
    ot_distance = ot.emd(a, b, M)
    return np.sum(ot_distance*M)


In [ ]:
for KO in KOIds:
    print(KO)
    otDists =  pd.DataFrame(np.zeros((50)))
    
    for z in range(50):
        otDists.loc[z] = ot_distance(targetCells.X, 
                                     adataKOScreen[adataKOScreen.obs["guide_id"] == KO].X,
                                     N=50)
    #print(otDists)
    myDF.loc[KO,"OTDist"]=np.float(otDists.mean(axis=0))

In [ ]:
myDF = myDF.sort_values("OTDist")

In [ ]:
myDF.to_csv('KO_otDists.csv', index=False)

In [ ]:
mySelConditions=list(myDF["KO"][0:20]) + list(["NTC", "IRF1", "FOXN1", "ZNF440" ]) 

In [ ]:
adataSel = adata[[x in mySelConditions for x in adata.obs["guide_id"]],:]

In [ ]:
adataSel

In [ ]:
del adataSel.uns['log1p']

In [ ]:
sc.tl.rank_genes_groups(adataSel, groupby='guide_id', reference="NTC",use_raw=True)

In [ ]:
mySelConditions

In [ ]:
sc.pl.rank_genes_groups(adataSel)

In [ ]:
sc.pl.dotplot(adataSel,myGeneList,
              groupby='guide_id', dendrogram=True, standard_scale='var' )

In [ ]:
sc.pl.heatmap(adataSel, myGeneList, groupby='guide_id', show_gene_labels=True)


In [ ]:
import umap
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
labels_numeric = le.fit_transform(adataSel.obs["guide_id"])

embedding = umap.UMAP(min_dist=0.1 ).fit_transform(X=adataSel.X, y=np.array(labels_numeric))

In [ ]:
labels_numeric

In [ ]:
np.array(adataSel.obs["guide_id"])

In [ ]:
classes = ["G_1", "G_CONTROL", "G_9", "G_8", "G_7", "G_6", "G_5", "G_4", "G_3", "G_2", "G_10"]
fig, ax = plt.subplots(1, figsize=(18, 9))
plt.scatter(*embedding.T, s=0.7, cmap='tab20', alpha=0.7,  )
#plt.ylim(6, 16)
#plt.xlim(5, 15)
#plt.setp(ax, xticks=[], yticks=[])
cbar = plt.colorbar(boundaries=np.arange(12)-0.5)
cbar.set_ticks(np.arange(11))
cbar.set_ticklabels(classes)
plt.title('Clusters 0, 1, 4, 7 Single KO Cells Embedded Via supervised UMAP based on perturbation module');